# 📘 Class Activity 3: DFA Transition Engine with Execution Tracing
### Course: Theory of Automata  |  Topic 3: DFA Transition Engine
**Total Marks: 10** (Question 1: 5 marks, Question 2: 5 marks)

**Instructions for students:**
- Read the concept explanation below carefully before attempting the questions.
- Write your code only inside the sections marked `# TODO: write your code here`.
- Do **not** modify the provided helper code (the `DFA` class and demo) or the test-case cells.
- Run each test-case cell after implementing the corresponding function.
- Run the **Main** cell at the end to see your functions in action.
- At the very end, add your own 2–3 line conceptual summary in a new Text cell.

## 🧠 Concept Explanation

**What is a DFA?**

A **Deterministic Finite Automaton (DFA)** is the simplest machine that can decide whether a string belongs to a language. It reads the input one symbol at a time and, for every (state, symbol) pair, there is **exactly one** next state. Formally, a DFA is a 5-tuple:

$$M = (Q, \Sigma, \delta, q_0, F)$$

| Symbol | Meaning | Python representation |
|---|---|---|
| $Q$ | finite set of states | `set` of strings, e.g. `{'q0','q1','q2'}` |
| $\Sigma$ | input alphabet | `set` of characters, e.g. `{'0','1'}` |
| $\delta$ | transition function $Q \times \Sigma \to Q$ | `dict` mapping `(state, symbol)` → `next_state` |
| $q_0$ | start state | a string, e.g. `'q0'` |
| $F$ | set of accepting (final) states | `set` of strings, e.g. `{'q2'}` |

**How a DFA processes a string:**
1. Start in state $q_0$.
2. For each symbol of the input, look up $\delta(\text{current state}, \text{symbol})$ and move to that state.
3. When the input is finished, **accept** if the current state is in $F$, otherwise **reject**.

**Why tracing matters:**
A DFA that only says "accepted / rejected" is a black box. An **execution trace** prints every step of the computation (e.g. `q0 --0--> q1`), which lets us see *exactly* how the machine reached its decision — essential for debugging automata and for explaining them.

**The DFA used in this activity** accepts binary strings over $\{0,1\}$ that **end with `01`**:

| State | on `0` | on `1` |
|---|---|---|
| → q0 (start) | q1 | q0 |
| q1 | q1 | q2 |
| *q2 (accepting)* | q1 | q0 |

## 🛠️ Provided Code For Understanding (Teacher Demo)

The cell below implements a **generic DFA class** using Python dictionaries to represent the 5-tuple `(Q, Σ, δ, q0, F)`. The `accepts` method evaluates a string but prints **nothing** about how it got there — that is what you will add in the questions below.

In [1]:
class DFA:
    """A generic DFA built from the 5-tuple (Q, Sigma, delta, q0, F)."""

    def __init__(self, states, alphabet, transitions, start_state, accept_states):
        self.Q = set(states)                 # Q      : set of states
        self.Sigma = set(alphabet)           # Sigma  : input alphabet
        self.delta = dict(transitions)       # delta  : {(state, symbol): next_state}
        self.q0 = start_state                # q0     : start state
        self.F = set(accept_states)          # F      : accepting states
        self._validate()

    def _validate(self):
        assert self.q0 in self.Q, "Start state must be in Q."
        assert self.F <= self.Q, "Accepting states must be a subset of Q."
        for (state, symbol), nxt in self.delta.items():
            assert state in self.Q and nxt in self.Q, "delta uses an unknown state."
            assert symbol in self.Sigma, "delta uses a symbol outside the alphabet."

    def accepts(self, input_string):
        """Return True if the DFA accepts input_string (no tracing)."""
        current = self.q0
        for symbol in input_string:
            if (current, symbol) not in self.delta:
                return False                 # undefined transition -> reject
            current = self.delta[(current, symbol)]
        return current in self.F

    def describe(self):
        print("Q     =", sorted(self.Q))
        print("Sigma =", sorted(self.Sigma))
        print("q0    =", self.q0)
        print("F     =", sorted(self.F))
        print("delta =")
        for (state, symbol), nxt in sorted(self.delta.items()):
            print(f"   delta({state}, {symbol}) = {nxt}")


# DFA that accepts binary strings ending with "01"
ends_with_01 = DFA(
    states={'q0', 'q1', 'q2'},
    alphabet={'0', '1'},
    transitions={
        ('q0', '0'): 'q1', ('q0', '1'): 'q0',
        ('q1', '0'): 'q1', ('q1', '1'): 'q2',
        ('q2', '0'): 'q1', ('q2', '1'): 'q0',
    },
    start_state='q0',
    accept_states={'q2'},
)

ends_with_01.describe()

print("\nQuick check with the plain accepts() method (no trace):")
for s in ["01", "0101", "110", ""]:
    print(f"  accepts({s!r:7}) -> {ends_with_01.accepts(s)}")

Q     = ['q0', 'q1', 'q2']
Sigma = ['0', '1']
q0    = q0
F     = ['q2']
delta =
   delta(q0, 0) = q1
   delta(q0, 1) = q0
   delta(q1, 0) = q1
   delta(q1, 1) = q2
   delta(q2, 0) = q1
   delta(q2, 1) = q0

Quick check with the plain accepts() method (no trace):
  accepts('01'   ) -> True
  accepts('0101' ) -> True
  accepts('110'  ) -> False
  accepts(''     ) -> False


---
## ❓ Question 1 (5 Marks)

Implement the function **`trace_dfa(dfa, input_string, verbose=True)`** which runs a DFA on a string **and records/prints every state transition**. It takes:
- `dfa`: a `DFA` object (as defined above),
- `input_string`: the string to process, and
- `verbose`: if `True`, print each step while processing (if `False`, print nothing).

It must return a tuple **`(accepted, trace)`** where:
- `accepted` is `True`/`False` (whether the DFA accepts the string), and
- `trace` is a **list of `(current_state, symbol, next_state)` tuples**, one per transition taken, in order.

**Required procedure:**
1. Start at `dfa.q0`.
2. For each symbol, look up `dfa.delta[(current_state, symbol)]` and append `(current_state, symbol, next_state)` to `trace`.
3. If `verbose` is `True`, print each step in the format `Step k: q0 --0--> q1`.
4. If there is **no transition** defined for `(current_state, symbol)` (e.g. a symbol outside the alphabet), stop immediately and return `(False, trace)` — do **not** add that failed step to `trace`.
5. After the last symbol, `accepted` is `True` only if the final state is in `dfa.F`. If `verbose`, also print `Final state: qX -> ACCEPTED` (or `REJECTED`).

**Example:** `trace_dfa(ends_with_01, "01")` should return
`(True, [('q0', '0', 'q1'), ('q1', '1', 'q2')])` and, with `verbose=True`, print:
```
Input: '01'   Start state: q0
  Step 1: q0 --0--> q1
  Step 2: q1 --1--> q2
  Final state: q2 -> ACCEPTED
```

In [7]:
def trace_dfa(dfa, input_string, verbose=True):
    """
    Run `dfa` on `input_string` and record every state transition.

    Parameters
    ----------
    dfa : DFA
        A DFA object (see the provided DFA class above).
    input_string : str
        The string to process, e.g. "0101"
    verbose : bool
        If True, print every transition step as it happens.

    Returns
    -------
    tuple(bool, list of tuple)
        (accepted, trace) where trace is a list of
        (current_state, symbol, next_state) tuples, e.g.
        [('q0', '0', 'q1'), ('q1', '1', 'q2')]
    """
    # TODO: write your code here
def trace_dfa(dfa, input_string, verbose=True):
    current_state = dfa.q0
    trace = []

    if verbose:
        print(f"Input: '{input_string}' Start state: {current_state}")

    for k, symbol in enumerate(input_string, start=1):
        key = (current_state, symbol)

        if key not in dfa.delta:
            return False, trace

        next_state = dfa.delta[key]
        trace.append((current_state, symbol, next_state))

        if verbose:
            print(f"Step {k}: {current_state} --{symbol}--> {next_state}")

        current_state = next_state

    accepted = current_state in dfa.F

    if verbose:
        result = "ACCEPTED" if accepted else "REJECTED"
        print(f"Final state: {current_state} -> {result}")

    return accepted, trace

### ✅ Test Cases — Question 1
Run this cell after implementing `trace_dfa`. If every `assert` passes silently, the implementation is correct.

In [8]:
# --- TA / Auto-grading test cases for Question 1 ---
test_cases_q1 = [
    ("01",   True,  [('q0', '0', 'q1'), ('q1', '1', 'q2')]),
    ("0101", True,  [('q0', '0', 'q1'), ('q1', '1', 'q2'), ('q2', '0', 'q1'), ('q1', '1', 'q2')]),
    ("1100", False, [('q0', '1', 'q0'), ('q0', '1', 'q0'), ('q0', '0', 'q1'), ('q1', '0', 'q1')]),
    ("",     False, []),
    ("0102", False, [('q0', '0', 'q1'), ('q1', '1', 'q2'), ('q2', '0', 'q1')]),
]

passed = 0
for s, exp_acc, exp_trace in test_cases_q1:
    result = trace_dfa(ends_with_01, s, verbose=False)
    ok = (result == (exp_acc, exp_trace))
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] trace_dfa({s!r}) -> {result}")

print(f"\n{passed}/{len(test_cases_q1)} test cases passed for Question 1.")
assert passed == len(test_cases_q1), "Question 1: some test cases failed."
print("\nVisual check of your printed trace for '0101':")
trace_dfa(ends_with_01, "0101", verbose=True)

[PASS] trace_dfa('01') -> (True, [('q0', '0', 'q1'), ('q1', '1', 'q2')])
[PASS] trace_dfa('0101') -> (True, [('q0', '0', 'q1'), ('q1', '1', 'q2'), ('q2', '0', 'q1'), ('q1', '1', 'q2')])
[PASS] trace_dfa('1100') -> (False, [('q0', '1', 'q0'), ('q0', '1', 'q0'), ('q0', '0', 'q1'), ('q1', '0', 'q1')])
[PASS] trace_dfa('') -> (False, [])
[PASS] trace_dfa('0102') -> (False, [('q0', '0', 'q1'), ('q1', '1', 'q2'), ('q2', '0', 'q1')])

5/5 test cases passed for Question 1.

Visual check of your printed trace for '0101':
Input: '0101' Start state: q0
Step 1: q0 --0--> q1
Step 2: q1 --1--> q2
Step 3: q2 --0--> q1
Step 4: q1 --1--> q2
Final state: q2 -> ACCEPTED


(True,
 [('q0', '0', 'q1'), ('q1', '1', 'q2'), ('q2', '0', 'q1'), ('q1', '1', 'q2')])

---
## ❓ Question 2 (5 Marks)

Implement the function **`compute_run_statistics(dfa, strings)`** which takes:
- `dfa`: a `DFA` object, and
- `strings`: a list of input strings,

and returns a tuple **`(acceptance_rate, accepted_count, total_steps)`** where:
- `accepted_count` = number of strings in `strings` that the DFA accepts,
- `total_steps` = total number of transitions executed, i.e. the sum of `len(trace)` over all strings, and
- `acceptance_rate` = `accepted_count / len(strings)`, **rounded to 4 decimal places**. If `strings` is empty, return `(0.0, 0, 0)`.

**Hint:** call your `trace_dfa` function from Question 1 on every string with `verbose=False` (so nothing is printed), and accumulate the counts.

In [9]:
def compute_run_statistics(dfa, strings):
    """
    Run a DFA over many strings and summarise the results.

    Parameters
    ----------
    dfa : DFA
        A DFA object.
    strings : list of str
        The input strings to process, e.g. ["01", "110", ""]

    Returns
    -------
    tuple(float, int, int)
        (acceptance_rate rounded to 4 dp, accepted_count, total_steps)
    """
    # TODO: write your code here
def compute_run_statistics(dfa, strings):
    if len(strings) == 0:
        return (0.0, 0, 0)

    accepted_count = 0
    total_steps = 0

    for input_string in strings:
        accepted, trace = trace_dfa(dfa, input_string, verbose=False)

        if accepted:
            accepted_count += 1

        total_steps += len(trace)

    acceptance_rate = round(accepted_count / len(strings), 4)

    return (acceptance_rate, accepted_count, total_steps)


### ✅ Test Cases — Question 2
Run this cell after implementing `compute_run_statistics`. If every `assert` passes silently, the implementation is correct.

In [10]:
# --- TA / Auto-grading test cases for Question 2 ---
sample_strings = ["01", "1101", "110", "", "0011", "101", "0102", "1001"]
expected_rate = 0.5
expected_accepted = 4
expected_steps = 23

rate, accepted_count, total_steps = compute_run_statistics(ends_with_01, sample_strings)

print(f"Strings          : {sample_strings}")
print(f"accepted_count   -> got {accepted_count}, expected {expected_accepted}")
print(f"total_steps      -> got {total_steps}, expected {expected_steps}")
print(f"acceptance_rate  -> got {rate}, expected {expected_rate}")

assert accepted_count == expected_accepted, "accepted_count is incorrect."
assert total_steps == expected_steps, "total_steps is incorrect."
assert abs(rate - expected_rate) < 1e-3, "acceptance_rate is incorrect."
assert compute_run_statistics(ends_with_01, []) == (0.0, 0, 0), "Empty list must return (0.0, 0, 0)."
print("\nAll Question 2 test cases passed!")

Strings          : ['01', '1101', '110', '', '0011', '101', '0102', '1001']
accepted_count   -> got 4, expected 4
total_steps      -> got 23, expected 23
acceptance_rate  -> got 0.5, expected 0.5

All Question 2 test cases passed!


---
## 🚀 Main — Demonstration

This cell prints the DFA, traces two full runs step by step (Question 1), and then summarises a batch of strings (Question 2) so you can see the whole pipeline end-to-end.

In [11]:
# ===== MAIN DEMO =====
print("### Step 1: The DFA (5-tuple) ###")
ends_with_01.describe()

print("\n### Step 2: Execution traces (Question 1) ###")
for w in ["10101", "1100"]:
    accepted, trace = trace_dfa(ends_with_01, w, verbose=True)
    print(f"  -> {len(trace)} transitions recorded\n")

print("### Step 3: Batch statistics (Question 2) ###")
batch = ["01", "1101", "110", "", "0011", "101", "0102", "1001"]
rate, accepted_count, total_steps = compute_run_statistics(ends_with_01, batch)
print(f"  Strings         : {batch}")
print(f"  Accepted        : {accepted_count} of {len(batch)}")
print(f"  Total steps     : {total_steps}")
print(f"  Acceptance rate : {rate}")

### Step 1: The DFA (5-tuple) ###
Q     = ['q0', 'q1', 'q2']
Sigma = ['0', '1']
q0    = q0
F     = ['q2']
delta =
   delta(q0, 0) = q1
   delta(q0, 1) = q0
   delta(q1, 0) = q1
   delta(q1, 1) = q2
   delta(q2, 0) = q1
   delta(q2, 1) = q0

### Step 2: Execution traces (Question 1) ###
Input: '10101' Start state: q0
Step 1: q0 --1--> q0
Step 2: q0 --0--> q1
Step 3: q1 --1--> q2
Step 4: q2 --0--> q1
Step 5: q1 --1--> q2
Final state: q2 -> ACCEPTED
  -> 5 transitions recorded

Input: '1100' Start state: q0
Step 1: q0 --1--> q0
Step 2: q0 --1--> q0
Step 3: q0 --0--> q1
Step 4: q1 --0--> q1
Final state: q1 -> REJECTED
  -> 4 transitions recorded

### Step 3: Batch statistics (Question 2) ###
  Strings         : ['01', '1101', '110', '', '0011', '101', '0102', '1001']
  Accepted        : 4 of 8
  Total steps     : 23
  Acceptance rate : 0.5


---
## 📝 Your Conceptual Understanding

A DFA processes an input string one symbol at a time and deterministically moves between states according to its transition function.Execution tracing shows each transition, making it easier to understand and verify how the DFA reaches its final decision. The acceptance rate shows the proportion of input strings accepted by the DFA, while total steps represents the transitions executed across all strings.